In [1]:
!pip install langchain_huggingface
!pip install langchain_community
!pip install arxiv
!pip install sentence-transformers
!pip install faiss-cpu
#!pip install numpy==1.26.4

  Using cached huggingface_hub-0.36.2-py3-none-any.whl.metadata (15 kB)
Using cached huggingface_hub-0.36.2-py3-none-any.whl (566 kB)
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.5.0
    Uninstalling huggingface_hub-1.5.0:
      Successfully uninstalled huggingface_hub-1.5.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
transformers 5.0.0 requires huggingface-hub<2.0,>=1.3.0, but you have huggingface-hub 0.36.2 which is incompatible.
  Using cached huggingface_hub-1.5.0-py3-none-any.whl.metadata (13 kB)
Using cached huggingface_hub-1.5.0-py3-none-any.whl (596 kB)
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 0.36.2
    Uninstalling huggingface_hub-0.36.2:
      Successfully uninstalled huggingface_hub-0.36.2
ERROR: pip's dependency resolver does not currently take into ac

In [2]:
from langchain_huggingface import HuggingFaceEmbeddings
EMB = HuggingFaceEmbeddings(
    model_name='sentence-transformers/all-MiniLM-L6-v2',
    model_kwargs=
    {
        'device': 'cuda'
    }
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [3]:
!CMAKE_ARGS="-DLLAMA_CUBLAS=on" pip install llama-cpp-python==0.2.77 -U --force-reinstall --no-cache-dir --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu121

Looking in indexes: https://pypi.org/simple, https://abetlen.github.io/llama-cpp-python/whl/cu121
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 326.6/326.6 MB 288.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 189.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.9/134.9 kB 27.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 261.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.6/44.6 kB 306.7 MB/s eta 0:00:00
  Attempting uninstall: typing-extensions
    Found existing installation: typing_extensions 4.15.0
    Uninstalling typing_extensions-4.15.0:
      Successfully uninstalled typing_extensions-4.15.0
  Attempting uninstall: numpy
    Found existing installation: numpy 1.26.4
    Uninstalling numpy-1.26.4:
      Successfully uninstalled numpy-1.26.4
  Attempting uninstall: MarkupSafe
    Found existing installation: MarkupSafe 3.0.3
    Uninstalling MarkupSafe-3.0.3:
      Successfully uninstalled

In [4]:
from langchain_core.prompts import PromptTemplate
from langchain_community.llms import LlamaCpp
from langchain_community.chat_models import ChatLlamaCpp
from langchain_core.callbacks import CallbackManager, StreamingStdOutCallbackHandler
from arxiv import Search
import pandas as pd
import numpy as np
import os
import json
import warnings
import logging
import time
import psutil


In [5]:
warnings.filterwarnings("ignore")

# Configure logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

In [8]:
from huggingface_hub import hf_hub_download

# Llama-2-7B-Chat
model_name = "llama-2-7b-chat.Q4_K_M.gguf"
model_repo_id = "TheBloke/Llama-2-7B-Chat-GGUF"
model_path_folder = "./models"


os.makedirs(model_path_folder, exist_ok=True)

# Download
llm_model_path = hf_hub_download(
    repo_id=model_repo_id,
    filename=model_name,
    local_dir=model_path_folder,
    local_dir_use_symlinks=False
)

print(f"New model downloaded to: {llm_model_path}")

New model downloaded to: models/llama-2-7b-chat.Q4_K_M.gguf


In [9]:
class QAbot():
    def __init__(self, prompt, Embeddings):
        self.EMBEDDINGS = Embeddings
        self.custom_prompt_template = prompt
        self.callback_manager = CallbackManager([StreamingStdOutCallbackHandler()])
        self.llm = self.load_llm()
        self.refer = ""


    def set_custom_prompt(self, context: str, question: str):
        """
        Create a prompt using the provided context and question.
        """
        prompt_template = PromptTemplate(template=self.custom_prompt_template, input_variables=['context', 'question'])
        prompt = prompt_template.format(context=context, question=question)
        logger.info("The prompt was created.")
        return prompt


    def fetch_arxiv_data(self, query):
        search = Search(
            query=query,
            max_results=1,
        )
        papers = list(search.results())
        sorted_papers = sorted(papers, key=lambda paper: paper.published, reverse=True)  # Sort by publication date
        logger.info("The data retrieved was created.")
        return sorted_papers


    def load_llm(self):
        """
        Loading the LLM using Llama Cpp.
        """
        llm = LlamaCpp(
            model_path=llm_model_path,
            callback_manager = self.callback_manager,
            n_ctx=3072,
            max_tokens = 4096, #idk if this is ok
            n_gpu_layers=-1,
            verbose=False,
            flash_attn=True,
            chat_format = None
        )
        logger.info("LLM model loaded.")
        print("LLM Model Loaded")
        return llm


    def qa_bot(self, question: str):
        """
        Manually perform the search, create the prompt, and get the response from the LLM.
        """
        arxiv_data = self.fetch_arxiv_data(question)
        context = ""
        for papers in arxiv_data:
            context += "**" + papers.title + "**\n"
            context += papers.summary + "\n\n"

        prompt = self.set_custom_prompt(context, question)
        print(f"Prompt\n{prompt}\n-----------------------------------")
        return self.llm.invoke(prompt)

    def get_reference(self):
        return self.refer

In [14]:
prompt = """
    based on the following metadata from arxiv.org
    {context}

    Summarize the research paper into the following four chunks, providing a brief summary of the main ideas and key points for each:
    ## 1. Introduction
    -
    ## 2. Discussion
    -
    ## 3. Conclusion
    -
    ## 4. Further Reading
    -
"""

qa_instance = QAbot(prompt, EMB)

LLM Model Loaded


In [15]:
def main():
    start_time = time.time()
    ans = []
    question = input("Enter your question below\n")

    final = ""
    l = []
    for chunk in qa_instance.qa_bot(question):
        l.append(time.time())
        final += str(chunk)

    print(final)
    _o = qa_instance.get_reference()
    ans.append([question, final, _o])

In [17]:
if __name__ == "__main__":
    main()

Enter your question below
Mao Zedong
Prompt

    based on the following metadata from arxiv.org
    **Limit Theory of the Multi-set Allocation Occupancy (MAO) Distribution: Normal and Poisson Approximations via MAO Norm**
This paper investigates the asymptotic behavior of the Multi-set Allocation Occupancy (MAO) distribution, which models the count vector $X=(X_{=0},\ldots,X_{=T})$ from $T$ independent rounds of sampling without replacement of size $m$ from $N$ individuals. Focusing on $X_{=t}$ (individuals in exactly $t$ subsets) and employing the MAO norm -- a combinatorial tool yielding closed-form factorial moments -- we derive the exact marginal distribution of a single individual as $\mathrm{Bin}(T,p)$ with $p=m/N$. Using the MAO norm, we prove that for any fixed number of distinct individuals, their joint distribution differs from the product of marginals by $O(1/N)$, establishing the weak dependence required for limit theorems. Based on these findings, we delineate two asymptot